In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product

import time
import sys
import requests
import logging
import os

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from scipy import stats
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from statsmodels.tsa.api import SimpleExpSmoothing, Holt, ExponentialSmoothing

In [2]:
df_fd = pd.read_excel("steron no BTM agc 19 Dec 25.xlsx", sheet_name="non-steron no BTM", skiprows=4)

In [3]:
print(df_fd)

                                              Brc   Agc      P/N  \
0                                              20  19.0   100192   
1                                              20  19.0   100478   
2                                              20  19.0   100918   
3                                              20  19.0   101732   
4                                              20  19.0   101918   
...                                           ...   ...      ...   
31305                                          98  19.0  5538724   
31306                                          98  19.0  5594377   
31307                                          98  19.0  5633611   
31308                                          98  19.0    60985   
31309  Printed : 01/12/2025 15:56:36, by Haryanto   NaN      NaN   

                         Desc  DN Price DR/\nNDR    OH   OO  Book  Alloc\nIn  \
0       SHAFT,FUEL PUMP DRIVE     33.31       DR   3.0  0.0   0.0        0.0   
1                 SEAL,

In [4]:
TARGET_BRANCH = "25"   #UBAH CABANGNYA
logging.info("BEGIN Constructing Selected Branch Data and Combining It to DF")

# Convert column names to lowercase
df_fd.columns = df_fd.columns.str.lower()

# Extract demand columns
demand_columns = sorted(
    [col for col in df_fd.columns if col.startswith("d-")],
    key=lambda x: int(x.split("-")[1]),
    reverse=True
)

# Normalize P/N
df_fd["p/n"] = df_fd["p/n"].str.upper()

# ===== FILTER BRANCH =====
df_fd_branch = df_fd[df_fd["brc"] == TARGET_BRANCH]
# If multiple branches:
# df_branch = df[df["branch"].isin(TARGET_BRANCH)]

# ===== GROUP & AGGREGATE =====
df_all = (
    df_fd_branch
    .groupby(["agc", "p/n"], as_index=False)[demand_columns]
    .sum()
)

# Convert summed demand values into list
df_all["d"] = df_all[demand_columns].values.tolist()

# Keep relevant columns
df_all = df_all[["agc", "p/n", "d"]]

# Insert branch label
df_all.insert(0, "branch", f"{TARGET_BRANCH}")

# Append aggregated data
df_fd = pd.concat([df_fd, df_all], ignore_index=True)

logging.info(
    f"Branch {TARGET_BRANCH} Data Constructed And Merged With DF | Total Rows: {len(df_fd)}"
)
print(df_all)


    branch   agc           p/n  \
0       25  19.0        100099   
1       25  19.0        100716   
2       25  19.0        107981   
3       25  19.0        108172   
4       25  19.0        109080   
..     ...   ...           ...   
985     25  19.0  S   923    E   
986     25  19.0       S 16002   
987     25  19.0       S 16052   
988     25  19.0       S 16054   
989     25  19.0       S 16069   

                                                     d  
0    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
1    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
2    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
3    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
4    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
..                                                 ...  
985  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
986  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
987  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  
988  [0.0, 0.0, 0.0, 

In [5]:
logging.info("BEGIN Mean, Std, UB Calculation, and Construct Clipping Data")

# Get mean and standard deviation of 12 periods before the last one
df_all["d"] = df_all["d"].apply(lambda x: x if isinstance(x, list) else [])  # Ensure d is a list
df_all['mean_12'] = df_all['d'].apply(lambda x: np.mean(x[-13:-1]))  # Use 12 periods before the last one
df_all['std_12'] = df_all['d'].apply(lambda x: np.std(x[-13:-1]))    # Use 12 periods before the last one

# Get upper bound from mean and std
df_all['ub'] = df_all['mean_12'] + 1.5 * df_all['std_12']

# Limit the original df to upper bound (using the 12 periods before the last one)
df_all['clipped_d'] = df_all.apply(lambda row: np.clip(row['d'][-13:-1], 0, row['ub']).tolist(), axis=1)

# Display the updated DataFrame
display(df_all)


,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d
0,25,19.0,100099,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,25,19.0,100716,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,25,19.0,107981,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,25,19.0,108172,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,25,19.0,109080,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...,...,...,...,...
985,25,19.0,S 923 E,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.166667,0.552771,0.995823,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.995..."
986,25,19.0,S 16002,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
987,25,19.0,S 16052,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.166667,0.372678,0.725684,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.7256836610416..."
988,25,19.0,S 16054,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.083333,0.276385,0.497911,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.497..."


In [6]:
logging.info("BEGIN Moving Average Calculation")

# Calculate Simple Moving Average
df_all['clipped_d_15'] = df_all.apply(lambda row: np.clip(row['d'][:15], 0, row['ub']).tolist(), axis=1)

# Function to compute SMA forecasts for D-13 to D-1 using 3-point averages
def sma_forecast(data):
    sma_values = []
    for i in range(13):  # We want 13 forecast points: D-13 to D-1
        window = data[i:i+3]
        forecast = np.mean(window)  # Equal weights
        sma_values.append(forecast)
    return sma_values

# Apply SMA forecasting logic
df_all['ma'] = df_all['clipped_d_15'].apply(sma_forecast)

# Extract the last forecast (for D-1)
df_all['ma_result'] = df_all['ma'].apply(lambda x: x[-1])

print(df_all)

    branch   agc           p/n  \
0       25  19.0        100099   
1       25  19.0        100716   
2       25  19.0        107981   
3       25  19.0        108172   
4       25  19.0        109080   
..     ...   ...           ...   
985     25  19.0  S   923    E   
986     25  19.0       S 16002   
987     25  19.0       S 16052   
988     25  19.0       S 16054   
989     25  19.0       S 16069   

                                                     d   mean_12    std_12  \
0    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
1    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
2    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
3    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
4    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
..                                                 ...       ...       ...   
985  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [7]:
import numpy as np
import pandas as pd

logging.info("BEGIN Weighted Moving Average Calculation")


# Function to compute WMA forecasts for D-13 to D-1
def wma_forecast_with_weights(data, weights):
    wma_values = []
    for i in range(13):  # Forecasting D-13 to D-1 using D-16 to D-2
        window = data[i:i+3]
        forecast = np.sum(np.array(window) * weights) / sum(weights)
        wma_values.append(forecast)
    return wma_values

# Define step size
step = 0.05

# Initialize columns to store best weights, forecasts, and WMA results
df_all['wma_best_w1'] = np.nan
df_all['wma_best_w2'] = np.nan
df_all['wma_best_w3'] = np.nan
df_all['wma_result'] = np.nan
df_all['wma_forecast'] = df_all.apply(lambda _: [], axis=1)  # Initialize as empty lists

# Optimize weights for each row
for idx, row in df_all.iterrows():
    best_rmse = float('inf')
    best_weights = (0.15, 0.25, 0.6)  # Initial weight assumption
    best_forecast = None
    best_full_forecast = None  # Store full forecast array

    # Iterate over valid w1 values
    for w1 in np.round(np.arange(0.15, 0.81, step), 2):  # w1 ≥ 0.15
        for w2 in np.round(np.arange(0.25, 0.86 - w1, step), 2):  # w2 ≥ 0.25 and w1 + w2 ≤ 0.85
            w3 = 1 - (w1 + w2)  # Ensure sum is exactly 1

            # Ensure w3 > w2 > w1
            if w3 > w2 > w1:
                weights = (w1, w2, w3)

                # Compute WMA forecast for this row
                wma_forecast = wma_forecast_with_weights(row['clipped_d_15'], weights)

                # Extract the D-1 prediction (last forecast)
                wma_result = wma_forecast[-1]

                # Extract actual last value of 'd' (D-1)
                d_last = row['d'][-1]

                # Compute RMSE for this row
                rmse = np.sqrt((d_last - wma_result) ** 2)

                # Store best weights if RMSE improves
                if rmse < best_rmse:
                    best_rmse = rmse
                    best_weights = weights
                    best_forecast = wma_result
                    best_full_forecast = wma_forecast  # Store full forecast

    # Store the best weights and forecast for this row
    df_all.at[idx, 'wma_best_w1'] = best_weights[0]
    df_all.at[idx, 'wma_best_w2'] = best_weights[1]
    df_all.at[idx, 'wma_best_w3'] = best_weights[2]
    df_all.at[idx, 'wma_result'] = best_forecast
    df_all.at[idx, 'wma_forecast'] = best_full_forecast  # Store full WMA forecast
    
print(df_all)


    branch   agc           p/n  \
0       25  19.0        100099   
1       25  19.0        100716   
2       25  19.0        107981   
3       25  19.0        108172   
4       25  19.0        109080   
..     ...   ...           ...   
985     25  19.0  S   923    E   
986     25  19.0       S 16002   
987     25  19.0       S 16052   
988     25  19.0       S 16054   
989     25  19.0       S 16069   

                                                     d   mean_12    std_12  \
0    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
1    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
2    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
3    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
4    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
..                                                 ...       ...       ...   
985  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [8]:
alpha_ewma = 0.4
def custom_exponential_weighted_moving_average(values, alpha=alpha_ewma):
    ewma_values = [values[0]]  # Start with the first value

    # Apply EWMA formula up to D-2 (i.e., index 11 if length = 12)
    for t in range(1, len(values)):
        if np.isnan(values[t]):
            ewma_t = alpha * 0 + (1 - alpha) * ewma_values[-1]
        else:
            ewma_t = alpha * values[t] + (1 - alpha) * ewma_values[-1]
        ewma_values.append(ewma_t)

    return ewma_values  # This gives you EWMA from D-13 to D-2


def ewma_forecast(data, alpha=alpha_ewma):
    # Calculate EWMA up to D-2
    ewma_up_to_d2 = custom_exponential_weighted_moving_average(data, alpha)

    # Forecast D-1 as same as EWMA at D-2
    ewma_d1 = ewma_up_to_d2[-1]

    # Append D-1 forecast to the EWMA list
    ewma_with_d1 = ewma_up_to_d2 + [ewma_d1]

    # Return full EWMA list (D-13 to D-1) and D-1 forecast
    return ewma_with_d1, ewma_d1

df_all['ewma'], df_all['ewma_result'] = zip(*df_all['clipped_d'].apply(lambda x: ewma_forecast(x[-12:], alpha_ewma)))
print(df_all)


    branch   agc           p/n  \
0       25  19.0        100099   
1       25  19.0        100716   
2       25  19.0        107981   
3       25  19.0        108172   
4       25  19.0        109080   
..     ...   ...           ...   
985     25  19.0  S   923    E   
986     25  19.0       S 16002   
987     25  19.0       S 16052   
988     25  19.0       S 16054   
989     25  19.0       S 16069   

                                                     d   mean_12    std_12  \
0    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
1    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
2    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
3    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
4    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
..                                                 ...       ...       ...   
985  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [9]:
logging.info("BEGIN Linear Reggression Calculation")

#LINEAR REGRESSION
#  Calculate Linear Regression
def lr(x):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)
    model =  LinearRegression()
    model.fit(df_all[['x']], df_all['y'])
    df_all.loc[len(df_all), 'x'] = len(df_all) + 1
    return model.predict(df_all[['x']])

df_all['lr'] = df_all['clipped_d'].apply(lambda x: lr(x).tolist())
df_all['lr_result'] = df_all['lr'].apply(lambda x: x[-1:])
print(df_all)

    branch   agc           p/n  \
0       25  19.0        100099   
1       25  19.0        100716   
2       25  19.0        107981   
3       25  19.0        108172   
4       25  19.0        109080   
..     ...   ...           ...   
985     25  19.0  S   923    E   
986     25  19.0       S 16002   
987     25  19.0       S 16052   
988     25  19.0       S 16054   
989     25  19.0       S 16069   

                                                     d   mean_12    std_12  \
0    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
1    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
2    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
3    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
4    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
..                                                 ...       ...       ...   
985  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [10]:
logging.info("BEGIN Polynomial Reggression Calculation")

#POLYNOMIAL 2ND AND 3RD
# Calculate Polynomial Regression
def pr(x, pr_degree):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)

    X = df_all[['x']]  # Independent variable (reshape to 2D array)
    y = df_all['y']    # Dependent variable

    poly = PolynomialFeatures(degree=pr_degree)  # Create polynomial features
    X_poly = poly.fit_transform(X)  # Transform input features
    poly_model = LinearRegression()  # Initialize linear regression model
    poly_model.fit(X_poly, y)  # Fit polynomial model

    df_all.loc[len(df_all), 'x'] = len(df_all) + 1
    X_all_poly = poly.transform(df_all[['x']])
    return poly_model.predict(X_all_poly)  

df_all['pr2'] = df_all['clipped_d'].apply(lambda x: pr(x, 2).tolist())
df_all['pr2_result'] = df_all['pr2'].apply(lambda x: x[-1:])
df_all['pr3'] = df_all['clipped_d'].apply(lambda x: pr(x, 3).tolist())
df_all['pr3_result'] = df_all['pr3'].apply(lambda x: x[-1:])
print(df_all)

    branch   agc           p/n  \
0       25  19.0        100099   
1       25  19.0        100716   
2       25  19.0        107981   
3       25  19.0        108172   
4       25  19.0        109080   
..     ...   ...           ...   
985     25  19.0  S   923    E   
986     25  19.0       S 16002   
987     25  19.0       S 16052   
988     25  19.0       S 16054   
989     25  19.0       S 16069   

                                                     d   mean_12    std_12  \
0    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
1    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
2    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
3    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
4    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
..                                                 ...       ...       ...   
985  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [11]:
logging.info("BEGIN Simple Exponential Smoothing Calculation")

alpha_ses = 0.8  # ubah nilai alpha (semakin besar semakin berat ke data terbaru)

#SES
def ses(x, alpha = alpha_ses):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)
    df_all.loc[len(df_all), 'x'] = len(df_all) + 1

    new_data = SimpleExpSmoothing(df_all['y']).fit(smoothing_level=alpha, optimized=False).fittedvalues
    return new_data.tolist()

df_all['ses'] = df_all['clipped_d'].apply(lambda x: ses(x, alpha_ses))
df_all['ses_result'] = df_all['ses'].apply(lambda x: x[-1:])
print(df_all)


    branch   agc           p/n  \
0       25  19.0        100099   
1       25  19.0        100716   
2       25  19.0        107981   
3       25  19.0        108172   
4       25  19.0        109080   
..     ...   ...           ...   
985     25  19.0  S   923    E   
986     25  19.0       S 16002   
987     25  19.0       S 16052   
988     25  19.0       S 16054   
989     25  19.0       S 16069   

                                                     d   mean_12    std_12  \
0    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
1    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
2    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
3    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
4    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
..                                                 ...       ...       ...   
985  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [12]:
logging.info("BEGIN Double Exponential Smoothing Calculation")

ALPHA = 0.1
BETA = 0.1

def des(x, alpha=ALPHA, beta=BETA):
    df_tmp = pd.DataFrame()
    df_tmp['y'] = x
    df_tmp['x'] = range(1, len(df_tmp) + 1)
    df_tmp.loc[len(df_tmp), 'x'] = len(df_tmp) + 1

    model = ExponentialSmoothing(
        df_tmp['y'],
        trend='add',
        seasonal=None
    )

    fitted_model = model.fit(
        smoothing_level=alpha,
        smoothing_trend=beta,
        optimized=False
    )

    return fitted_model.fittedvalues.tolist()
df_all['des'] = df_all['clipped_d'].apply(lambda x: des(x))
df_all['des_result'] = df_all['des'].apply(lambda x: x[-1])
print(df_all)

    branch   agc           p/n  \
0       25  19.0        100099   
1       25  19.0        100716   
2       25  19.0        107981   
3       25  19.0        108172   
4       25  19.0        109080   
..     ...   ...           ...   
985     25  19.0  S   923    E   
986     25  19.0       S 16002   
987     25  19.0       S 16052   
988     25  19.0       S 16054   
989     25  19.0       S 16069   

                                                     d   mean_12    std_12  \
0    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
1    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
2    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
3    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
4    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
..                                                 ...       ...       ...   
985  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [13]:
# Calculate metrics including MASE, MAPE, and SMAPE
def metric(x):
    period_length = len(x['clipped_d'])
    df_all = pd.DataFrame()
    df_all['qty'] = x['clipped_d'][:period_length]  # Ground truth values
    
    # Naive forecast (previous period's value)
    df_all['naive'] = df_all['qty'].shift(1)

    models = ['ma', 'wma_forecast', 'ewma', 'lr', 'pr2', 'pr3', 'ses', 'des']
    for model in models:
        df_all[model] = x[model][:period_length]

    # Compute MASE scaling factor (denominator)
    naive_diff = np.abs(df_all['qty'].diff()).dropna()
    naive_mae = naive_diff.mean() if not naive_diff.empty else np.nan

    result = []
    for model in models:
        y_true = df_all['qty'].dropna()
        y_pred = df_all[model].dropna()
        y_naive = df_all['naive'].dropna()

        # Standard error metrics
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)

        # Relative errors for MdRAE and GMRAE
        relative_errors = np.abs(y_true - y_pred) / np.abs(y_true - y_naive)
        relative_errors = relative_errors.replace([np.inf, -np.inf], np.nan).dropna()

        # Compute MdRAE and GMRAE
        if not relative_errors.empty:
            mdrae = np.median(relative_errors)
            gmrae = np.exp(np.mean(np.log(relative_errors)))
        else:
            mdrae, gmrae = np.nan, np.nan

        # Compute MASE
        mase = mae / naive_mae if naive_mae > 0 else np.nan

        # Compute MAPE (bounded between 0% - 100%)
        mape_values = np.abs((y_true - y_pred) / y_true)
        mape_values = mape_values.replace([np.inf, -np.inf], np.nan).dropna()
        mape = 100 * mape_values.mean() if not mape_values.empty else np.nan

        # Compute SMAPE (bounded between 0% - 100%)
        smape_values = np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-10)  # Avoid div by zero
        smape_values = smape_values.replace([np.inf, -np.inf], np.nan).dropna()
        smape = 100 * smape_values.mean() if not smape_values.empty else np.nan

        result.append({
            'model': model, 'RMSE': rmse, 'MAE': mae, 'R2': r2,
            'MdRAE': mdrae, 'GMRAE': gmrae, 'MASE': mase, 'MAPE': mape, 'SMAPE': smape
        })

    metrics_df_all = pd.DataFrame(result)

    # Select the best model based on MAE
    best_model_row = metrics_df_all.loc[metrics_df_all['MAE'].idxmin()]
    best_model = best_model_row['model']

    return {'best_model': best_model, 'metrics': metrics_df_all.to_dict(orient='records')}

# Apply metric function
df_all['metric'] = df_all.apply(lambda x: metric(x), axis=1)

# Extract best model and metrics
df_all['best_model'] = df_all['metric'].apply(lambda x: x['best_model'])
df_all['metrics'] = df_all['metric'].apply(lambda x: x['metrics'])
df_all = df_all.drop(columns=['metric'])
# Define the number of months
num_months = 13

# Create new columns dynamically for each month
for i in range(num_months, 0, -1):
    df_all[f'pred_{i}'] = df_all.apply(
        lambda x: x[x['best_model']][num_months - i] if pd.notna(x['best_model']) else np.nan, axis=1
    )

# Extract R² of the best model into a new column
def get_best_model_r2(row):
    best_model = row['best_model']
    for m in row['metrics']:
        if m['model'] == best_model:
            return m['R2']
    return np.nan

df_all['best_r2'] = df_all.apply(get_best_model_r2, axis=1)
# Mark R2 performance
df_all['note'] = np.where(df_all['best_r2'] < 0.25, "R2 < 0.25", "Good")
print(df_all)



c:\Users\Brandon\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Brandon\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


    branch   agc           p/n  \
0       25  19.0        100099   
1       25  19.0        100716   
2       25  19.0        107981   
3       25  19.0        108172   
4       25  19.0        109080   
..     ...   ...           ...   
985     25  19.0  S   923    E   
986     25  19.0       S 16002   
987     25  19.0       S 16052   
988     25  19.0       S 16054   
989     25  19.0       S 16069   

                                                     d   mean_12    std_12  \
0    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
1    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
2    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
3    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
4    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
..                                                 ...       ...       ...   
985  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [14]:
#kalkulasi semua model D-0
logging.info("BEGIN Data Selection Calculation")
# Select the best model for each row
df_all['mean_12_FD'] = df_all['d'].apply(lambda x: np.mean(x[-12:]))
df_all['std_12_FD'] = df_all['d'].apply(lambda x: np.std(x[-12:]))
df_all['ub_FD'] = df_all['mean_12_FD'] + 1.5 * df_all['std_12_FD']
df_all['clipped_d_FD'] = df_all.apply(lambda row: np.clip(row['d'][-12:], 0, row['ub_FD']).tolist(), axis=1)
print(df_all)

    branch   agc           p/n  \
0       25  19.0        100099   
1       25  19.0        100716   
2       25  19.0        107981   
3       25  19.0        108172   
4       25  19.0        109080   
..     ...   ...           ...   
985     25  19.0  S   923    E   
986     25  19.0       S 16002   
987     25  19.0       S 16052   
988     25  19.0       S 16054   
989     25  19.0       S 16069   

                                                     d   mean_12    std_12  \
0    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
1    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
2    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
3    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
4    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
..                                                 ...       ...       ...   
985  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [15]:
logging.info("BEGIN Moving Average Calculation")

# Calculate Simple Moving Average
df_all['clipped_d_15_FD'] = df_all.apply(lambda row: np.clip(row['d'][-15:], 0, row['ub_FD']).tolist(), axis=1)

# Function to compute SMA forecasts for D-13 to D-1 using 3-point averages
def sma_forecast(data):
    sma_values = []
    for i in range(13):  # We want 13 forecast points: D-13 to D-1
        window = data[i:i+3]
        forecast = np.mean(window)  # Equal weights
        sma_values.append(forecast)
    return sma_values

# Apply SMA forecasting logic
df_all['ma_FD'] = df_all['clipped_d_15_FD'].apply(sma_forecast)

# Extract the last forecast (for D-1)
df_all['ma_result_FD'] = df_all['ma_FD'].apply(lambda x: x[-1])

print(df_all)

    branch   agc           p/n  \
0       25  19.0        100099   
1       25  19.0        100716   
2       25  19.0        107981   
3       25  19.0        108172   
4       25  19.0        109080   
..     ...   ...           ...   
985     25  19.0  S   923    E   
986     25  19.0       S 16002   
987     25  19.0       S 16052   
988     25  19.0       S 16054   
989     25  19.0       S 16069   

                                                     d   mean_12    std_12  \
0    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
1    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
2    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
3    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
4    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
..                                                 ...       ...       ...   
985  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [16]:
import numpy as np
import pandas as pd

logging.info("BEGIN Weighted Moving Average Calculation for FD")

# Function to compute WMA forecasts for D-13 to D-1
def wma_forecast_with_weights_FD(data, weights):
    wma_values_FD = []
    for i in range(13):  # Forecasting D-13 to D-1 using D-16 to D-2
        window_FD = data[i:i+3]
        forecast_FD = np.sum(np.array(window_FD) * weights) / sum(weights)
        wma_values_FD.append(forecast_FD)
    return wma_values_FD

# Define step size
step_FD = 0.05

# Initialize columns to store best weights, forecasts, and WMA results
df_all['wma_best_w1_FD'] = np.nan
df_all['wma_best_w2_FD'] = np.nan
df_all['wma_best_w3_FD'] = np.nan
df_all['wma_result_FD'] = np.nan
df_all['wma_forecast_FD'] = df_all.apply(lambda _: [], axis=1)  # Initialize as empty lists

# Optimize weights for each row
for idx, row in df_all.iterrows():
    best_rmse_FD = float('inf')
    best_weights_FD = (0.15, 0.25, 0.6)  # Initial weight assumption
    best_forecast_FD = None
    best_full_forecast_FD = None  # Store full forecast array

    # Iterate over valid w1_FD values
    for w1_FD in np.round(np.arange(0.15, 0.81, step_FD), 2):  # w1_FD ≥ 0.15
        for w2_FD in np.round(np.arange(0.25, 0.86 - w1_FD, step_FD), 2):  # w2_FD ≥ 0.25 and w1_FD + w2_FD ≤ 0.85
            w3_FD = 1 - (w1_FD + w2_FD)  # Ensure sum is exactly 1

            # Ensure w3_FD > w2_FD > w1_FD
            if w3_FD > w2_FD > w1_FD:
                weights_FD = (w1_FD, w2_FD, w3_FD)

                # Compute WMA forecast for this row
                wma_forecast_FD = wma_forecast_with_weights_FD(row['clipped_d_15_FD'], weights_FD)

                # Extract the D-1 prediction (last forecast)
                wma_result_FD = wma_forecast_FD[-1]

                # Extract actual last value of 'd' (D-1)
                d_last_FD = row['d'][-1]

                # Compute RMSE for this row
                rmse_FD = np.sqrt((d_last_FD - wma_result_FD) ** 2)

                # Store best weights if RMSE improves
                if rmse_FD < best_rmse_FD:
                    best_rmse_FD = rmse_FD
                    best_weights_FD = weights_FD
                    best_forecast_FD = wma_result_FD
                    best_full_forecast_FD = wma_forecast_FD  # Store full forecast

    # Store the best weights and forecast for this row
    df_all.at[idx, 'wma_best_w1_FD'] = best_weights_FD[0]
    df_all.at[idx, 'wma_best_w2_FD'] = best_weights_FD[1]
    df_all.at[idx, 'wma_best_w3_FD'] = best_weights_FD[2]
    df_all.at[idx, 'wma_result_FD'] = best_forecast_FD
    df_all.at[idx, 'wma_forecast_FD'] = best_full_forecast_FD  # Store full WMA forecast
print(df_all)

    branch   agc           p/n  \
0       25  19.0        100099   
1       25  19.0        100716   
2       25  19.0        107981   
3       25  19.0        108172   
4       25  19.0        109080   
..     ...   ...           ...   
985     25  19.0  S   923    E   
986     25  19.0       S 16002   
987     25  19.0       S 16052   
988     25  19.0       S 16054   
989     25  19.0       S 16069   

                                                     d   mean_12    std_12  \
0    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
1    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
2    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
3    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
4    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
..                                                 ...       ...       ...   
985  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [17]:
# EWMA
alpha_ewma = 0.4

# Custom Exponential Weighted Moving Average Function
def custom_exponential_weighted_moving_average(values, alpha=alpha_ewma):
    ewma_values = [values[0]]  # Start with the first value (D-12)

    # Apply the EWMA formula for D-11 to D-1 (i.e., 11 more steps)
    for t in range(1, len(values)):  # len(values) = 12
        if np.isnan(values[t]):
            ewma_t = alpha * 0 + (1 - alpha) * ewma_values[-1]
        else:
            ewma_t = alpha * values[t] + (1 - alpha) * ewma_values[-1]
        ewma_values.append(ewma_t)
    
    return ewma_values  # EWMA from D-12 to D-1

# Forecast Function Using the Custom EWMA
def ewma_forecast(data, alpha=alpha_ewma):
    # Compute EWMA values for D-12 to D-1
    ewma_values = custom_exponential_weighted_moving_average(data, alpha)

    # Forecast D-0 as the same as EWMA at D-1
    forecast_d0 = ewma_values[-1]

    # Full series includes D-12 to D-0 (13 values total)
    ewma_full = ewma_values + [forecast_d0]

    return ewma_full, forecast_d0

# Apply the EWMA forecast to the dataset
df_all['ewma_FD'], df_all['ewma_result_FD'] = zip(*df_all['clipped_d_FD'].apply(lambda x: ewma_forecast(x[-12:], alpha_ewma)))

print(df_all)


    branch   agc           p/n  \
0       25  19.0        100099   
1       25  19.0        100716   
2       25  19.0        107981   
3       25  19.0        108172   
4       25  19.0        109080   
..     ...   ...           ...   
985     25  19.0  S   923    E   
986     25  19.0       S 16002   
987     25  19.0       S 16052   
988     25  19.0       S 16054   
989     25  19.0       S 16069   

                                                     d   mean_12    std_12  \
0    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
1    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
2    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
3    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
4    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
..                                                 ...       ...       ...   
985  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [18]:
#LR
def lr(x):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)
    model =  LinearRegression()
    model.fit(df_all[['x']], df_all['y'])
    df_all.loc[len(df_all), 'x'] = len(df_all) + 1
    return model.predict(df_all[['x']])
df_all['lr_FD'] = df_all['clipped_d_FD'].apply(lambda x: lr(x).tolist())
df_all['lr_result_FD'] = df_all['lr_FD'].apply(lambda x: x[-1:])
print(df_all)

    branch   agc           p/n  \
0       25  19.0        100099   
1       25  19.0        100716   
2       25  19.0        107981   
3       25  19.0        108172   
4       25  19.0        109080   
..     ...   ...           ...   
985     25  19.0  S   923    E   
986     25  19.0       S 16002   
987     25  19.0       S 16052   
988     25  19.0       S 16054   
989     25  19.0       S 16069   

                                                     d   mean_12    std_12  \
0    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
1    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
2    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
3    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
4    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
..                                                 ...       ...       ...   
985  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [19]:
#PR2&3
def pr(x, pr_degree):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)
    X = df_all[['x']]  # Independent variable (reshape to 2D array)
    y = df_all['y']    # Dependent variable
    poly = PolynomialFeatures(degree=pr_degree)  # Create polynomial features
    X_poly = poly.fit_transform(X)  # Transform input features
    poly_model = LinearRegression()  # Initialize linear regression model
    poly_model.fit(X_poly, y)  # Fit polynomial model
    df_all.loc[len(df_all), 'x'] = len(df_all) + 1
    X_all_poly = poly.transform(df_all[['x']])
    return poly_model.predict(X_all_poly)  
df_all['pr2_FD'] = df_all['clipped_d_FD'].apply(lambda x: pr(x, 2).tolist())
df_all['pr2_result_FD'] = df_all['pr2_FD'].apply(lambda x: x[-1:])
df_all['pr3_FD'] = df_all['clipped_d_FD'].apply(lambda x: pr(x, 3).tolist())
df_all['pr3_result_FD'] = df_all['pr3_FD'].apply(lambda x: x[-1:])
display(df_all)

,branch,agc,p/n,d,mean_12,std_12,ub,clipped_d,clipped_d_15,ma,...,wma_result_FD,wma_forecast_FD,ewma_FD,ewma_result_FD,lr_FD,lr_result_FD,pr2_FD,pr2_result_FD,pr3_FD,pr3_result_FD
0,25,19.0,100099,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.896241,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.597494,"[-0.19150439697221466, -0.13405307788055026, -...",[0.4979114321277582],"[0.18466495422320522, 0.03693299084463947, -0....",[1.1203007222874617],"[-0.1313173007808952, 0.0656586503904959, 0.13...",[1.9916457285109015]
1,25,19.0,100716,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0]
2,25,19.0,107981,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0]
3,25,19.0,108172,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0]
4,25,19.0,109080,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",[0.0]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
985,25,19.0,S 923 E,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.166667,0.552771,0.995823,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.995...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.597...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.39832914...",0.051623,"[0.025533919596295324, 0.03597961397659794, 0....",[0.15088225215992673],"[-0.10669530688451946, -0.024124579878317648, ...",[-0.06789701347196864],"[0.01459081119786726, -0.035150590613090976, -...",[-0.4023526724264259]
986,25,19.0,S 16002,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,0.000000,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0

In [20]:
#SES
def ses(x, alpha = alpha_ses):
    df_all = pd.DataFrame()
    df_all['y'] = x
    df_all['x'] = range(1, len(df_all) + 1)
    df_all.loc[len(df_all), 'x'] = len(df_all) + 1
    new_data = SimpleExpSmoothing(df_all['y']).fit(smoothing_level=alpha, optimized=False).fittedvalues
    return new_data.tolist()
df_all['ses_FD'] = df_all['clipped_d_FD'].apply(lambda x: ses(x, alpha_ses))
df_all['ses_result_FD'] = df_all['ses_FD'].apply(lambda x: x[-1:])
print(df_all)

    branch   agc           p/n  \
0       25  19.0        100099   
1       25  19.0        100716   
2       25  19.0        107981   
3       25  19.0        108172   
4       25  19.0        109080   
..     ...   ...           ...   
985     25  19.0  S   923    E   
986     25  19.0       S 16002   
987     25  19.0       S 16052   
988     25  19.0       S 16054   
989     25  19.0       S 16069   

                                                     d   mean_12    std_12  \
0    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
1    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
2    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
3    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
4    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
..                                                 ...       ...       ...   
985  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [21]:
# DES - FD
logging.info("BEGIN Double Exponential Smoothing FD Calculation")

ALPHA_FD = 0.1
BETA_FD = 0.1

# Double Exponential Smoothing function for FD
def des_FD(x, alpha=ALPHA_FD, beta=BETA_FD):
    df_tmp = pd.DataFrame()
    df_tmp['y'] = x
    df_tmp['x'] = range(1, len(df_tmp) + 1)
    df_tmp.loc[len(df_tmp), 'x'] = len(df_tmp) + 1

    model = ExponentialSmoothing(
        df_tmp['y'],
        trend='add',
        seasonal=None
    )

    fitted_model = model.fit(
        smoothing_level=alpha,
        smoothing_trend=beta,
        optimized=False
    )

    return fitted_model.fittedvalues.tolist()
df_all['des_FD'] = df_all['clipped_d_FD'].apply(lambda x: des_FD(x))
df_all['des_result_FD'] = df_all['des_FD'].apply(lambda x: x[-1])


In [22]:
logging.info("BEGIN Metric Calculation for _FD")

# Calculate metrics including MdRAE, GMRAE, MASE, MAPE, and SMAPE
def metric_FD(x):
    period_length = len(x['clipped_d_FD'])
    df_all = pd.DataFrame()
    df_all['qty'] = x['clipped_d_FD'][:period_length]  # Ground truth values

    # Naive forecast (previous period's value)
    df_all['naive'] = df_all['qty'].shift(1)

    models = ['ma_FD', 'wma_forecast_FD', 'ewma_FD', 'lr_FD', 'pr2_FD', 'pr3_FD', 'ses_FD', 'des_FD']
    for model in models:
        df_all[model] = x[model][:period_length]

    # Compute MASE scaling factor (denominator)
    naive_diff = np.abs(df_all['qty'].diff()).dropna()
    naive_mae = naive_diff.mean() if not naive_diff.empty else np.nan

    result = []
    for model in models:
        y_true = df_all['qty'].dropna()
        y_pred = df_all[model].dropna()
        y_naive = df_all['naive'].dropna()

        # Standard error metrics
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)

        # Relative errors for MdRAE and GMRAE
        relative_errors = np.abs(y_true - y_pred) / np.abs(y_true - y_naive)
        relative_errors = relative_errors.replace([np.inf, -np.inf], np.nan).dropna()

        # Compute MdRAE and GMRAE
        if not relative_errors.empty:
            mdrae = np.median(relative_errors)
            gmrae = np.exp(np.mean(np.log(relative_errors)))
        else:
            mdrae, gmrae = np.nan, np.nan

        # Compute MASE
        mase = mae / naive_mae if naive_mae > 0 else np.nan

        # Compute MAPE (bounded between 0% - 100%)
        mape_values = np.abs((y_true - y_pred) / y_true)
        mape_values = mape_values.replace([np.inf, -np.inf], np.nan).dropna()
        mape = 100 * mape_values.mean() if not mape_values.empty else np.nan

        # Compute SMAPE (bounded between 0% - 100%)
        smape_values = np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-10)  # Avoid div by zero
        smape_values = smape_values.replace([np.inf, -np.inf], np.nan).dropna()
        smape = 100 * smape_values.mean() if not smape_values.empty else np.nan

        result.append({
            'model': model, 'RMSE': rmse, 'MAE': mae, 'R2': r2,
            'MdRAE': mdrae, 'GMRAE': gmrae, 'MASE': mase, 'MAPE': mape, 'SMAPE': smape
        })

    return result  # Returning the metrics list

# Apply the metric function
df_all['metrics_FD'] = df_all.apply(lambda x: metric_FD(x), axis=1)

def get_best_r2_FD(row):
    best_model = row['best_model']
    metrics_fd = row.get('metrics_FD', [])
    for m in metrics_fd:
        if m['model'] == best_model + '_FD':
            return m['R2']
    return np.nan
df_all['best_r2_FD'] = df_all.apply(get_best_r2_FD, axis=1)
df_all['r2_status_FD'] = np.where(df_all['best_r2_FD'] < 0.25, "R2 < 0.25", "Good")

print(df_all)

c:\Users\Brandon\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Brandon\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Brandon\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


    branch   agc           p/n  \
0       25  19.0        100099   
1       25  19.0        100716   
2       25  19.0        107981   
3       25  19.0        108172   
4       25  19.0        109080   
..     ...   ...           ...   
985     25  19.0  S   923    E   
986     25  19.0       S 16002   
987     25  19.0       S 16052   
988     25  19.0       S 16054   
989     25  19.0       S 16069   

                                                     d   mean_12    std_12  \
0    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
1    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
2    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
3    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
4    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
..                                                 ...       ...       ...   
985  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [23]:
def apply_best_model_forecast(row):
    best_model = row['best_model']
    if best_model == 'ma':
        return row['ma_result_FD']
    elif best_model == 'wma':
        return row['wma_result']
    elif best_model == 'ewma':
        return row['ewma_result_FD']
    elif best_model == 'lr':
        return row['lr_result_FD'][-1] if isinstance(row['lr_result_FD'], list) else row['lr_result_FD']
    elif best_model == 'pr2':
        return row['pr2_result_FD'][-1] if isinstance(row['pr2_result_FD'], list) else row['pr2_result_FD']
    elif best_model == 'pr3':
        return row['pr3_result_FD'][-1] if isinstance(row['pr3_result_FD'], list) else row['pr3_result_FD']
    elif best_model == 'ses':
        return row['ses_result_FD'][-1] if isinstance(row['ses_result_FD'], list) else row['ses_result_FD']
    elif best_model == 'des':
        return row['des_result_FD'][-1] if isinstance(row['des_result_FD'], list) else row['des_result_FD']
    else:
        return np.nan
    
df_all['FD_forecast'] = df_all.apply(apply_best_model_forecast, axis=1)
# Define the number of months (from 12 to 1, excluding 0)
num_months = 13  # Total months (D-12 to D-0), but we exclude D-0

# Map best model to the correct forecast series (excluding pred_0_FD)
def extract_forecast_values(row, month_idx):
    best_model = row['best_model']
    forecast_column = f"{best_model}_FD"  # Example: 'ma_result_FD', 'wma_result_FD'
    
    if forecast_column in row and isinstance(row[forecast_column], list):
        if len(row[forecast_column]) >= (13 - month_idx):
            return row[forecast_column][12 - month_idx]  # Extract the correct past forecast
    return np.nan  # Return NaN if data is missing or not a list

# Create columns for pred_12_FD to pred_1_FD
for i in range(num_months - 1, 0, -1):  # From 12 to 1
    df_all[f'pred_{i}_FD'] = df_all.apply(lambda x: extract_forecast_values(x, i), axis=1)

# Ensure FD_forecast contains only numeric values
df_all['FD_final'] = np.maximum(0, df_all['FD_forecast'].round().astype(int))
# Get all columns except the last four we want to reorder
columns_to_keep = [col for col in df_all.columns if col not in ['best_model', 'metrics', 'FD_forecast', 'FD_final']]

# Define the new order with the last four columns at the end
column_order = columns_to_keep + ['best_model', 'metrics', 'FD_forecast', 'FD_final']

# Reorder DataFrame
df_all = df_all[column_order]

print(df_all)


    branch   agc           p/n  \
0       25  19.0        100099   
1       25  19.0        100716   
2       25  19.0        107981   
3       25  19.0        108172   
4       25  19.0        109080   
..     ...   ...           ...   
985     25  19.0  S   923    E   
986     25  19.0       S 16002   
987     25  19.0       S 16052   
988     25  19.0       S 16054   
989     25  19.0       S 16069   

                                                     d   mean_12    std_12  \
0    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
1    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
2    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
3    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
4    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...  0.000000  0.000000   
..                                                 ...       ...       ...   
985  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [24]:
df = pd.read_excel("steron no BTM agc 19 Dec 25.xlsx", sheet_name="non-steron no BTM", skiprows=4)

# --- Normalize column names ---
df.columns = df.columns.str.lower()

# --- Normalize P/N ---
df["p/n"] = df["p/n"].astype(str).str.upper()

# --- Filter ONLY target branch ---
df_branch = df[df["brc"] == TARGET_BRANCH]

# --- Identify all C- columns ---
c_columns = sorted(
    [col for col in df_branch.columns if isinstance(col, str) and col.startswith("c-")],
    key=lambda x: int(x.split("-")[1]),
    reverse=True
)

# Required columns
oh_col = "oh"
oo_col = "oo"
dn_price_col = "dn price"

# --- Aggregation rules ---
aggregation_dict = {col: "sum" for col in c_columns + [oh_col, oo_col]}
aggregation_dict[dn_price_col] = "first"

# --- GROUP BY P/N ---
df_sum = (
    df_branch
    .groupby("p/n", as_index=False)
    .agg(aggregation_dict)
)

# --- Total Calls ---
df_sum["Total Calls"] = df_sum[c_columns].sum(axis=1)

# --- Drop individual C- columns ---
df_sum = df_sum.drop(columns=c_columns)

# --- Add identifiers ---
df_sum.insert(0, "Agc", "All")
df_sum.insert(0, "Brc", TARGET_BRANCH)

# --- Reorder columns ---
df_sum = df_sum[
    ["Brc", "Agc", "p/n", "Total Calls", "dn price", "oh", "oo"]
]


In [25]:
# ===============================
# Perhitungan RC (Rank Call)
# ===============================

# Sort by Total Calls descending
df_sum = df_sum.sort_values(
    by="Total Calls", ascending=False
).reset_index(drop=True)

# Add cumulative sum
df_sum["Accum."] = df_sum["Total Calls"].cumsum()

# Percentage cumulative
total_calls = df_sum["Accum."].iloc[-1]
df_sum["%Accum."] = (df_sum["Accum."] / total_calls) * 100
df_sum["%Accum."] = df_sum["%Accum."].round(2)

# Klasifikasi RC (ABCD)
def assign_rc(pct):
    if pct < 50:
        return "A"
    elif pct < 80:
        return "B"
    elif pct < 100:
        return "C"
    else:
        return "D"

df_sum["RC"] = df_sum["%Accum."].apply(assign_rc)


In [26]:
df_final = (
    df_all[["branch", "agc", "p/n", "FD_final"]]
    .merge(
        df_sum[["p/n", "RC"]],
        on="p/n",
        how="left"
    )
)

In [27]:
print(df_final)

    branch   agc           p/n  FD_final RC
0       25  19.0        100099         0  B
1       25  19.0        100716         0  D
2       25  19.0        107981         0  D
3       25  19.0        108172         0  D
4       25  19.0        109080         0  D
..     ...   ...           ...       ... ..
985     25  19.0  S   923    E         0  B
986     25  19.0       S 16002         0  D
987     25  19.0       S 16052         0  B
988     25  19.0       S 16054         0  B
989     25  19.0       S 16069         0  C

[990 rows x 5 columns]


In [28]:
df_final.to_excel(f"brc {TARGET_BRANCH} agc 19.xlsx", index=False)